In [ ]:
import requests
import json
from numpy.random import randint
from time import sleep
import os
from pathlib import Path
from datetime import datetime
from utils import *
from concurrent.futures import ThreadPoolExecutor
from typing import List
from math import ceil

## Pegando informações dos Anúncios

In [ ]:
bairros = [
    "Cidade Nova América",
    "Consolação",
    "Vila Ede",
    "Brás",
    "Jardim Miragaia",
    "Vila Seabra",
    "Jardim Colonial",
    "Jardim América",
    "Jardim Zilda",
    "Jardim Novo Horizonte"
]

In [ ]:
chunck_paginas_por_thread = 5
chunck_request = 100

In [ ]:
with open('historico.json') as f:
    historico = json.loads(f.read())

In [ ]:
codigo_pesquisa = str(len(historico) + 1).rjust(5, "0")

In [ ]:
codigo_pesquisa

In [ ]:
pasta_destino = f'dados/{codigo_pesquisa}'
create_folder(pasta_destino)

In [ ]:
body = {
    'sorted': [{
        'id': 'postedAt',
        'desc': True,
        'label': 'Mais Recente'
    }],
    'pageSize': chunck_request,
    'filtered': [
        {'id': 'cities', 'value': ['São Paulo']},
        # {'id': 'properties_type', 'value': [
        #     'Apartamento Padrão',
        #     'Quitinete',
        #     'Flat',
        #     'Cobertura'
        # ]},
        # {'id': "start_useable_area", 'value': "80"},
        # {'id': "end_useable_area", 'value': "130"},
        # {'id': 'business_type', 'value': ['Locação']},
        {'id': 'business_type', 'value': ['Venda']},
        # {'id': 'start_buy_price', 'value': '700_000'},
        # {'id': 'end_buy_price', 'value': '1_000_000'},
  ]
}

In [ ]:
futures = []
executor = ThreadPoolExecutor(max_workers = 8)

for bairro in bairros:
    submit_bairro(
        codigo_pesquisa,
        body,
        bairro,
        executor,
        futures
    )

In [ ]:
len([future for future in futures if not future.done()])

In [ ]:
body['filtered'].append({'id': 'districts', 'value': bairros})
historico[codigo_pesquisa] = body
with open('historico.json', 'w') as f:
    f.write(json.dumps(historico))

In [ ]:
(268 - 0) * chunck_paginas_por_thread

In [ ]:
1340 / 17

max_workers = 1 -> 4.5 chuncks por minuto
max_workers = 3 -> 9.5 chuncks por minuto
max_workers = 4 -> 13.0 chuncks por minuto
max_workers = 5 -> 11.5 chuncks por minuto
max_workers = 6 -> 10.7 chuncks por minuto

Por bairro:
max_workers = 4 -> 78.8 chuncks por minuto

## Pegando informação de Proprietários

In [ ]:
import requests
import json
from numpy.random import randint
from time import sleep
import os
from pathlib import Path
from datetime import datetime
from utils import *
from concurrent.futures import ThreadPoolExecutor
from typing import List
from math import ceil

In [ ]:
codigo_pesquisa = '00012'
prefixo = codigo_pesquisa
pasta_destino = f'dados/{codigo_pesquisa}'

In [ ]:
lista_anuncios = read_chunks(f'{codigo_pesquisa} - anuncios_*.json', fr'./dados/{codigo_pesquisa}')

In [ ]:
len(lista_anuncios)

In [ ]:
# for anuncio in lista_anuncios[ultimo_anuncio:]:
#     if ultimo_anuncio % 10 == 0:
#         with open(f'{pasta_destino}/{prefixo} - logs.txt', 'a', newline='\n') as f:
#             f.write(f'{str(datetime.now())} - {ultimo_anuncio} / {len(lista_anuncios)} anúncios lidos. {ultimo_anuncio - nao_encontrados} ok.\n')

#         update_file(f'{pasta_destino}/{prefixo} - proprietarios_anuncios.json', dados_proprietarios_anuncios)

#         with open('ultimo_anuncio.txt', 'w') as f:
#             f.write(str(ultimo_anuncio))    

#         with open('nao_encontrados.txt', 'w') as f:
#             f.write(str(nao_encontrados))    

#         update_file(f'{pasta_destino}/{prefixo} - erros.json', erros)

#         dados_proprietarios_anuncios = []
#         erros = []

#     headers.update(
#         {"Referer" : f"https://painel.fisgar.com.br/anuncios?aid={anuncio['id']}"}
#     )

#     try:
#         ultimo_anuncio += 1
#         response = requests.post(
#             url='https://painel.fisgar.com.br/api123/v1/api/region/sp/owners/GetPeople?page=0&size=100',
#             headers = headers,
#             timeout = 7,
#             json = {
#                 'announce_id': anuncio['id'],
#                 'search': {
#                     'type': 'IPTU_ADDRESS',
#                     'key': {
#                         'number': anuncio['number'],
#                         'zipcode': anuncio['zipcode'],
#                         'extra': anuncio['extra'],
#                         'city': anuncio['city'],
#                         'district': anuncio['district'],
#                         'state': '',
#                         'name': None,
#                         'street': anuncio['street']
#                     }
#                 }
#             }
#         )

#         if response.status_code != 200:
#             nao_encontrados += 1
#             erros.append(response.json())
#             continue

#         json_data = response.json()

#         json_data.update(
#             {'announce_id': anuncio['id']}
#         )

#         dados_proprietarios_anuncios.append(json_data)

#     except requests.exceptions.Timeout:
#         nao_encontrados += 1
#         erros.append('Timeout!')

#     except json.JSONDecodeError:
#         nao_encontrados += 1
#         erros.append('JSONDecodeError')


In [ ]:
contador = 0

executor = ThreadPoolExecutor(max_workers = 4)
futures = []

logs_path = Path(f'./dados/{codigo_pesquisa}/{codigo_pesquisa} - logs.txt')

if not logs_path.is_file():
    with open(logs_path, 'w') as f:
        f.write('')

lock_proprietarios = Lock()
lock_logs = Lock()
lock_erros = Lock()

for i in range(contador, len(lista_anuncios) // 10 + 1, 1):
    futures.append(executor.submit(
        pesquisar_dados_proprietarios_pelo_anuncio,
        lista_anuncios[i * 10: (i + 1) * 10],
        len(lista_anuncios),
        i,
        codigo_pesquisa,
        lock_proprietarios,
        lock_erros,
        lock_logs,
    ))

In [ ]:
futures

In [ ]:
{
    "_and":[
        {
            "$trigger": {
                "body": {
                    "event": {
                        "_eq": "invoice.status_changed"
                    }
                }
            }
        },
        {
            "$trigger": {
                "body": {
                    "data[status]": {
                        "_eq": "paid"
                    }
                }
            }
        },
    ]
}

In [ ]:
{
    "filter": {
        "_and" : [
            {
                "contrato": {
                    "_eq": "{{$trigger.body.keys[0]}}"
                }
            },
            {
                "vencimento": {
                    "_eq": "{{data_vencimento}}"
                }
            }
        ]
    },
    "aggregate": {
        "count": ["contrato"]
    }
}